::: {.claves}
[En breve · {{< meta tipo >}}]{.etiqueta}

{{< meta resumen >}}
:::

In [677]:
#| label: setup
#| include: false
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").exists())
sys.path.insert(0, str(ROOT))

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from common.paths import topic_dir, data_dirs
from common.files import read_processed
from common.plotting import ACCENT, GREY, PALETTE, init_plotly, new_figure, add_source
init_plotly()

# Tablas legibles: separadores en español, sin forma ni tipos
pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(20)
pl.Config.set_thousands_separator(".")
pl.Config.set_decimal_separator(",")
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_hide_column_data_types(True)

TOPIC = topic_dir()
RAW, PROCESSED = data_dirs(TOPIC)

## Preguntas

*(Pendiente: 2–3 líneas de por qué importa, con un dato de contexto que enganche.)*

1. **P1.** ¿Cómo ha evolucionado la población residente en España según su **nacionalidad** (española o extranjera)?
2. **P2.** ¿Cómo ha evolucionado según su **país de origen**, entendido como país de nacimiento (nacidos en España o en el extranjero y, dentro de estos, por región)?

Además, para interpretar ambas: ¿cuánto cambia el resultado según se defina «extranjero» por nacionalidad o por lugar de nacimiento?

## Datos y método

- **Fuente.** INE, Estadística Continua de Población (ECP), tabla 56938 [@ine_ecp_tabla_56938]: población residente por fecha, sexo, edad, nacionalidad y país de nacimiento. Datos descargados el 25 de octubre de 2026; la metodología está descrita en @ine_ecp_metodologia.
- **Unidad de observación: la persona residente.** Es residente quien, en la fecha de referencia, tiene su residencia habitual en España, y el INE entiende por residencia habitual el lugar de residencia *registrado* [@ine_ecp_metodologia]. Cada persona cuenta una vez. La cifra mide, por tanto, personas inscritas como residentes y no permisos de residencia, hogares ni movimientos de entrada y salida, y depende de cómo y cuándo se registran las altas y las bajas.
- **Cobertura.** 24 observaciones: población a 1 de enero, de 2002 a 2025. Total nacional; todos los sexos y edades. Se usa el 1 de enero porque es la única fecha disponible en toda la serie: hasta 2021 las estimaciones eran semestrales (1 de enero y 1 de julio) y desde 2021 son trimestrales [@ine_ecp_metodologia].
- **Definiciones.** *Nacionalidad*: vínculo jurídico entre una persona y un Estado, medido en la fecha de referencia; quien tiene nacionalidad española se clasifica como español aunque tenga otra, y los apátridas se incluyen en extranjera. *País de nacimiento*: país donde se produjo el nacimiento, con independencia de la nacionalidad; se distingue entre España y extranjero y, dentro de este, por región.
- **Qué se calcula.** Población, peso sobre el total, variación anual, media anual y tasa compuesta por periodo, y contribución de cada grupo al crecimiento. Las fórmulas están en el *Detalle técnico*.
- **Periodos.** Los cortes se fijaron antes de mirar los datos, por hitos de la economía española: 2002–2008 (expansión hasta el inicio de la crisis), 2008–2014 (crisis), 2014–2019 (recuperación, hasta el último año antes de la pandemia) y 2019–2025 (pandemia y después). Que las conclusiones no dependan de esa elección se comprueba en «Robustez», con bloques de 5 años.

In [678]:
#| label: carga
#| code-fold: true
df = read_processed(TOPIC, "poblacion_nacionalidad_origen")

# Series base: todos los sexos y todas las edades
base = df.filter(pl.col("sexo") == "Total", pl.col("edad") == "Todas las edades")


def ancho(d, columna):
    """Tabla ancha: una fila por año (anio) y una columna por categoría."""
    return (d.pivot(on=columna, index="periodo", values="total")
             .sort("periodo")
             .with_columns(anio=pl.col("periodo").dt.year())
             .drop("periodo")
             .select("anio", pl.exclude("anio")))


# P1: por nacionalidad
nac = ancho(base.filter(pl.col("pais_nacimiento") == "Total"), "nacionalidad")

# P2: por país de nacimiento
pais = ancho(base.filter(pl.col("nacionalidad") == "Total"), "pais_nacimiento")

# Cruce nacionalidad x lugar de nacimiento (cuatro grupos)
ESP_ESP = "Nacionalidad española · nacidos en España"
ESP_EXT = "Nacionalidad española · nacidos en el extranjero"
EXT_ESP = "Nacionalidad extranjera · nacidos en España"
EXT_EXT = "Nacionalidad extranjera · nacidos en el extranjero"
cruce = ancho(
    base.filter(pl.col("nacionalidad").is_in(["Española", "Extranjera"]),
                pl.col("pais_nacimiento").is_in(["España", "Extranjero"]))
        .with_columns(grupo=pl.col("nacionalidad") + "|" + pl.col("pais_nacimiento")),
    "grupo",
).rename({"Española|España": ESP_ESP, "Española|Extranjero": ESP_EXT,
          "Extranjera|España": EXT_ESP, "Extranjera|Extranjero": EXT_EXT})
cruce = cruce.with_columns(Total=pl.sum_horizontal(ESP_ESP, ESP_EXT, EXT_ESP, EXT_EXT))

assert nac["anio"].to_list() == pais["anio"].to_list() == cruce["anio"].to_list()
assert (cruce["Total"] - nac["Total"]).abs().max() <= 5

In [679]:
#| label: funciones
#| code-fold: true
PERIODOS = [(2002, 2008), (2008, 2014), (2014, 2019), (2019, 2025)]
PERIODOS_TOTAL = PERIODOS + [(2002, 2025)]
BLOQUES_5 = [(2002, 2007), (2007, 2012), (2012, 2017), (2017, 2022), (2022, 2025)]
ANIOS_CLAVE = [2002, 2008, 2014, 2019, 2025]

REG_NAC = ["Europa", "Sudamérica", "África", "Asia", "América Central y Caribe",
           "América del Norte", "Oceanía", "Apátridas"]
REG_PAIS = ["Europa", "Sudamérica", "África", "Asia", "América Central y Caribe",
            "América del Norte", "Oceanía"]

TINTA = "#4b5563"
COL = {"Española": GREY, "Extranjera": ACCENT, "España": GREY, "Extranjero": ACCENT, "Total": TINTA}
COL_REG = {"Europa": "#2563eb", "Sudamérica": ACCENT, "África": "#0d9488", "Asia": "#a16207",
           "América Central y Caribe": "#7c3aed", "América del Norte": "#db2777",
           "Oceanía": "#65a30d", "Apátridas": GREY}
FUENTE = "Fuente: INE (ECP, tabla 56938), población a 1 de enero. Elaboración propia."


def etiqueta(p):
    return f"{p[0]}–{p[1]}"


def txt_anio(d):
    """Año como texto para mostrarlo en tablas (sin separador de miles)."""
    return d.with_columns(pl.col("anio").cast(pl.String)).rename({"anio": "año"})


def variacion_anual(wide, cols):
    """Variación respecto al año anterior: absoluta (Δ) y en % (Δ%), por columna."""
    return wide.select(
        "anio",
        *[pl.col(c).diff().alias(f"Δ {c}") for c in cols],
        *[(100 * pl.col(c).pct_change()).round(2).alias(f"Δ% {c}") for c in cols],
    )


def resumen_periodos(wide, cols, total, periodos):
    """Por periodo y grupo: población inicial y final, variación, media anual,
    tasa anual compuesta (%) y contribución (%) a la variación del total."""
    idx = {a: i for i, a in enumerate(wide["anio"].to_list())}
    filas = []
    for a, b in periodos:
        var_total = wide[total][idx[b]] - wide[total][idx[a]]
        for c in [*cols, total]:
            p0, p1 = wide[c][idx[a]], wide[c][idx[b]]
            var = p1 - p0
            filas.append({
                "periodo": etiqueta((a, b)), "grupo": c, "inicial": p0, "final": p1,
                "variación": var, "media anual": round(var / (b - a)),
                "tasa compuesta (%)": round(100 * ((p1 / p0) ** (1 / (b - a)) - 1), 2),
                "contribución (%)": round(100 * var / var_total, 1),
            })
    return pl.DataFrame(filas)


def tablas_regiones(wide, regiones, total):
    """Cuatro tablas por región: niveles, peso sobre `total`, media anual y contribución por periodo."""
    clave = wide.filter(pl.col("anio").is_in(ANIOS_CLAVE))
    res = resumen_periodos(wide, regiones, total, PERIODOS_TOTAL)
    res = res.with_columns((100 * (pl.col("final") / pl.col("inicial") - 1)).round(1).alias("variación (%)"))
    orden = {etiqueta(p): i for i, p in enumerate(PERIODOS_TOTAL)}

    def por_periodo(valor):
        return (res.filter(pl.col("grupo") != total)
                   .pivot(on="grupo", index="periodo", values=valor)
                   .with_columns(o=pl.col("periodo").replace_strict(orden)).sort("o").drop("o")
                   .select("periodo", *regiones))

    return {
        "niveles": txt_anio(clave.select("anio", *regiones, total)),
        "peso": txt_anio(clave.select("anio", *[(100 * pl.col(r) / pl.col(total)).round(1).alias(r) for r in regiones])),
        "media": por_periodo("media anual"),
        "tasa": por_periodo("tasa compuesta (%)"),
        "cambio": por_periodo("variación (%)"),
        "contribucion": por_periodo("contribución (%)"),
    }


def fig_total(wide):
    anios = wide["anio"].to_list()
    fig = new_figure(y_title="Variación anual (personas)", height=440)
    pct = (100 * wide["Total"].pct_change()).to_list()
    fig.add_bar(x=anios, y=wide["Total"].diff().to_list(), name="Variación anual (personas y %)", marker_color=GREY,
                text=["" if p is None else ("0,0 %" if round(p, 1) == 0 else f"{p:+.1f} %".replace(".", ",").replace("-", "−")) for p in pct],
                textposition="outside", textfont=dict(size=11), cliponaxis=False, customdata=pct,
                hovertemplate="%{y:,.0f} personas (%{customdata:.2f} %)")
    fig.add_scatter(x=anios, y=wide["Total"].to_list(), name="Población total (eje derecho)",
                    mode="lines+markers", yaxis="y2", line=dict(color=ACCENT, width=3), marker=dict(size=5),
                    hovertemplate="%{y:,.0f}")
    fig.update_layout(yaxis2=dict(overlaying="y", side="right", showgrid=False, tickformat=",.0f",
                                  title="Población total", range=[40e6, 50e6], tick0=40e6, dtick=2e6),
                      yaxis=dict(range=[-300_000, 1_000_000]), xaxis=dict(dtick=2), margin=dict(r=80))
    add_source(fig, FUENTE)
    return fig


def fig_indice_peso(wide, nativo, foraneo, etiqueta_foraneo):
    """Tres paneles: población de cada grupo (millones), cambio acumulado desde 2002 (%) y peso del grupo extranjero."""
    anios = wide["anio"].to_list()
    miles = lambda v: f"{v:,.0f}".replace(",", ".")
    fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.08,
                        subplot_titles=("Población (millones)", "Cambio desde 2002 (%)", f"Peso de {etiqueta_foraneo} (%)"))
    y_max = 0
    for g in (nativo, foraneo):
        # panel 1: niveles (millones); etiquetas en el primer y el último año
        mill = (wide[g] / 1e6).to_list()
        fig.add_scatter(x=anios, y=mill, name=g, mode="lines", line=dict(color=COL[g], width=3),
                        text=[f"{miles(p)} personas" for p in wide[g].to_list()], hovertemplate="%{text}", row=1, col=1)
        for i in (0, -1):
            fig.add_annotation(x=anios[i], y=mill[i], text=f"{mill[i]:.1f} M".replace(".", ","), showarrow=False,
                               xanchor="left" if i == 0 else "right", xshift=6 if i == 0 else -4,
                               yshift=-16 if (i == 0 and g == foraneo) else 14,
                               font=dict(color=COL[g]), row=1, col=1)
        # panel 2: cambio acumulado (%)
        cambio = (100 * (wide[g] / wide[g][0] - 1)).to_list()
        y_max = max(y_max, max(cambio))
        # texto del hover construido aquí (así el redondeo no depende del formato de Plotly)
        hover = [f"{c:+.0f} % · {miles(p)} personas" for c, p in zip(cambio, wide[g].to_list())]
        fig.add_scatter(x=anios, y=cambio, name=g, mode="lines", line=dict(color=COL[g], width=3), showlegend=False,
                        text=hover, hovertemplate="%{text}", row=1, col=2)
        v = cambio[-1]
        txt = (f"{v:+.1f}" if abs(v) < 10 else f"{v:+.0f}").replace(".", ",")
        fig.add_annotation(x=anios[-1], y=v, text=f"{txt} %", showarrow=False,
                           xanchor="right", yshift=14, font=dict(color=COL[g]), row=1, col=2)
    peso = (100 * wide[foraneo] / wide["Total"]).to_list()
    fig.add_scatter(x=anios, y=peso, name=f"Peso de {etiqueta_foraneo}", mode="lines",
                    line=dict(color=ACCENT, width=3), showlegend=False, hovertemplate="%{y:.1f} %", row=1, col=3)
    for i in (0, -1):
        fig.add_annotation(x=anios[i], y=peso[i], text=f"{peso[i]:.1f} %".replace(".", ","), showarrow=False,
                           xanchor="left" if i == 0 else "right", xshift=8 if i == 0 else -4,
                           yshift=-14 if i == 0 else 14, font=dict(color=ACCENT), row=1, col=3)
    fig.update_yaxes(tickformat=".0f", ticksuffix=" M", rangemode="tozero", row=1, col=1)
    fig.update_yaxes(tickformat=",.0f", ticksuffix=" %", zeroline=True, zerolinecolor="rgba(127,127,127,.5)",
                     range=[-0.05 * y_max, 1.12 * y_max], row=1, col=2)
    fig.update_yaxes(tickformat=".0f", ticksuffix=" %", rangemode="tozero", row=1, col=3)
    fig.update_xaxes(dtick=8)
    fig.update_layout(height=440, margin=dict(l=60, r=40, t=110, b=55), hovermode="x unified",
                      legend=dict(y=1.22))
    add_source(fig, FUENTE)
    return fig


def fig_variacion_anual(wide, nativo, foraneo):
    """Barras apiladas de la variación anual (personas); el Δ% de cada grupo (como en la tabla) va en el hover."""
    anios = wide["anio"].to_list()
    fig = new_figure(y_title="Variación anual (personas)", height=440, barmode="relative")

    def pct(g):
        return (100 * wide[g].pct_change()).to_list()

    def fmt(v):
        return "" if v is None or v != v else f"{v:+.2f}".replace(".", ",") + " %"

    for g in (nativo, foraneo):
        d = wide[g].diff().to_list()
        p = [fmt(v) for v in pct(g)]
        hover = [f"{'' if x is None else format(x, '+,.0f').replace(',', '.')} personas ({t})" for x, t in zip(d, p)]
        fig.add_bar(x=anios, y=d, name=g, marker_color=COL[g], hovertext=hover, hovertemplate="%{hovertext}")
    d = wide["Total"].diff().to_list()
    p = [fmt(v) for v in pct("Total")]
    hover = [f"{'' if x is None else format(x, '+,.0f').replace(',', '.')} personas ({t})" for x, t in zip(d, p)]
    fig.add_scatter(x=anios, y=d, name="Total", mode="lines+markers",
                    line=dict(color=TINTA, width=2), marker=dict(size=5), hovertext=hover, hovertemplate="%{hovertext}")
    fig.update_layout(xaxis=dict(dtick=1))
    add_source(fig, FUENTE)
    return fig


def fig_periodos(res, grupos, titulo_y="Variación media anual (personas por año)"):
    """Barras apiladas de la media anual por periodo; el rombo es el total. El hover resume media, tasa y contribución."""
    fig = new_figure(y_title=titulo_y, height=440, barmode="relative")
    num = lambda v, f: format(v, f).replace(",", "X").replace(".", ",").replace("X", ".")
    for g in grupos:
        d = res.filter(pl.col("grupo") == g)
        hover = [f"{num(m, '+,.0f')}/año · {num(t, '+.1f')} % anual · {num(c, '.0f')} % del total"
                 for m, t, c in zip(d["media anual"], d["tasa compuesta (%)"], d["contribución (%)"])]
        fig.add_bar(x=d["periodo"].to_list(), y=d["media anual"].to_list(), name=g,
                    marker_color=COL.get(g, COL_REG.get(g)), hovertext=hover, hovertemplate="%{hovertext}")
    t = res.filter(pl.col("grupo") == "Total")
    hover = [f"{format(m, '+,.0f').replace(',', '.')}/año · {format(r, '+.1f').replace('.', ',')} % anual"
             for m, r in zip(t["media anual"], t["tasa compuesta (%)"])]
    fig.add_scatter(x=t["periodo"].to_list(), y=t["media anual"].to_list(), name="Total", mode="markers",
                    marker=dict(symbol="diamond", size=11, color=TINTA, line=dict(color="white", width=1)),
                    hovertext=hover, hovertemplate="%{hovertext}")
    fig.update_xaxes(type="category")
    add_source(fig, FUENTE)
    return fig


def fig_regiones(wide, regiones, total, nombre_total):
    """Tres vistas con botones: personas, peso (%) y variación media anual por periodo."""
    anios = wide["anio"].to_list()
    res = resumen_periodos(wide, regiones, total, PERIODOS_TOTAL)
    n = len(regiones)
    fig = new_figure(y_title="Personas", height=520, barmode="relative")
    for r in regiones:
        fig.add_scatter(x=anios, y=wide[r].to_list(), name=r, mode="lines",
                        line=dict(color=COL_REG[r], width=2.5), hovertemplate="%{y:,.0f}")
    for r in regiones:
        fig.add_scatter(x=anios, y=(100 * wide[r] / wide[total]).to_list(), name=r, mode="lines",
                        line=dict(color=COL_REG[r], width=2.5), hovertemplate="%{y:.1f} %", visible=False)
    tot = res.filter(pl.col("grupo") == total)
    num = lambda v, f: format(v, f).replace(",", "X").replace(".", ",").replace("X", ".")
    for r in regiones:
        d = res.filter(pl.col("grupo") == r)
        hover = [f"{num(m, '+,.0f')}/año · {num(c, '.0f')} % del total"
                 for m, c in zip(d["media anual"], d["contribución (%)"])]
        fig.add_bar(x=d["periodo"].to_list(), y=d["media anual"].to_list(), name=r, marker_color=COL_REG[r],
                    hovertext=hover, hovertemplate="%{hovertext}", visible=False)
    fig.add_scatter(x=tot["periodo"].to_list(), y=tot["media anual"].to_list(), name=nombre_total.capitalize(),
                    mode="markers", marker=dict(symbol="diamond", size=10, color=TINTA, line=dict(color="white", width=1)),
                    hovertext=[f"{num(t, '+,.0f')}/año (= 100 %)" for t in tot["media anual"]],
                    hovertemplate="%{hovertext}", visible=False)

    def vista(k, titulo, sufijo, formato, tipo_x):
        dtick = 2 if tipo_x == "linear" else 1
        visible = [i // n == k for i in range(3 * n)] + [k == 2]
        return dict(label=["Personas", "Peso (%)", "Variación media anual por periodo"][k], method="update",
                    args=[{"visible": visible},
                          {"yaxis.title.text": titulo, "yaxis.ticksuffix": sufijo, "yaxis.tickformat": formato,
                           "xaxis.type": tipo_x, "xaxis.dtick": dtick}])

    botones = [vista(0, "Personas", "", ",.0f", "linear"),
               vista(1, f"% sobre {nombre_total}", " %", ".1f", "linear"),
               vista(2, "Personas por año", "", ",.0f", "category")]
    fig.update_layout(updatemenus=[dict(type="buttons", direction="right", x=0, y=1.3, xanchor="left",
                                        yanchor="bottom", buttons=botones, showactive=True)],
                      margin=dict(t=120), xaxis=dict(dtick=2))
    add_source(fig, FUENTE)
    return fig


COL.update({"Nacionalidad española · nacidos en España": GREY, "Nacionalidad española · nacidos en el extranjero": "#2563eb",
            "Nacionalidad extranjera · nacidos en España": "#0d9488", "Nacionalidad extranjera · nacidos en el extranjero": ACCENT})


def _es(v, f):
    """Formato español (miles con punto, decimales con coma) construido a mano para el hover."""
    return format(v, f).replace(",", "X").replace(".", ",").replace("X", ".")


def fig_cruce_pesos(cr):
    """Barras apiladas: peso (%) sobre el total de los tres grupos que no son «nacionalidad española · nacidos en España»."""
    anios = cr["anio"].to_list()
    fig = new_figure(y_title="% sobre la población total", height=440, barmode="stack")
    for g in (ESP_EXT, EXT_ESP, EXT_EXT):
        pesos = (100 * cr[g] / cr["Total"]).to_list()
        fig.add_bar(x=anios, y=pesos, name=g, marker_color=COL[g],
                    hovertext=[f"{_es(p, '.1f')} % · {_es(n, ',.0f')} personas" for p, n in zip(pesos, cr[g].to_list())],
                    hovertemplate="%{hovertext}")
    fig.update_layout(xaxis=dict(dtick=2))
    fig.update_yaxes(ticksuffix=" %", tickformat=".0f")
    add_source(fig, FUENTE)
    return fig


def fig_definiciones(cr):
    """Dos paneles: población extranjera según cada definición (millones) y su peso sobre el total (%)."""
    anios = cr["anio"].to_list()
    series = {"Residentes con nacionalidad extranjera": cr[EXT_ESP] + cr[EXT_EXT],
              "Residentes nacidos en el extranjero": cr[ESP_EXT] + cr[EXT_EXT]}
    colores = {"Residentes con nacionalidad extranjera": ACCENT, "Residentes nacidos en el extranjero": "#2563eb"}
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                        subplot_titles=("Población «extranjera» según cada definición (millones)", "Peso sobre la población total (%)"))
    for nombre, ser in series.items():
        mill = (ser / 1e6).to_list()
        peso = (100 * ser / cr["Total"]).to_list()
        fig.add_scatter(x=anios, y=mill, name=nombre, mode="lines", line=dict(color=colores[nombre], width=3),
                        hovertext=[f"{_es(n, ',.0f')} personas" for n in ser.to_list()], hovertemplate="%{hovertext}", row=1, col=1)
        fig.add_scatter(x=anios, y=peso, name=nombre, mode="lines", line=dict(color=colores[nombre], width=3),
                        showlegend=False, hovertext=[f"{_es(p, '.1f')} %" for p in peso], hovertemplate="%{hovertext}", row=1, col=2)
        for col, y, txt in ((1, mill[-1], f"{_es(mill[-1], '.1f')} M"), (2, peso[-1], f"{_es(peso[-1], '.1f')} %")):
            fig.add_annotation(x=anios[-1], y=y, text=txt, showarrow=False, xanchor="right", yshift=14,
                               font=dict(color=colores[nombre]), row=1, col=col)
    fig.update_yaxes(rangemode="tozero", ticksuffix=" M", tickformat=".0f", row=1, col=1)
    fig.update_yaxes(rangemode="tozero", ticksuffix=" %", tickformat=".0f", row=1, col=2)
    fig.update_xaxes(dtick=4)
    fig.update_layout(height=440, margin=dict(l=60, r=40, t=110, b=55), hovermode="x unified", legend=dict(y=1.22))
    add_source(fig, FUENTE)
    return fig


def fig_explica(cr):
    """Cuatro paneles (2 x 2): las proporciones que explican la diferencia entre ambas definiciones."""
    anios = cr["anio"].to_list()
    paneles = [
        ("Nacionalidad española y nacidos fuera<br>(% de los de nacionalidad española)", cr[ESP_EXT], cr[ESP_ESP] + cr[ESP_EXT], COL[ESP_EXT]),
        ("Nacionalidad extranjera y nacidos en España<br>(% de los de nacionalidad extranjera)", cr[EXT_ESP], cr[EXT_ESP] + cr[EXT_EXT], COL[EXT_ESP]),
        ("Nacidos fuera con nacionalidad extranjera<br>(% de los nacidos fuera)", cr[EXT_EXT], cr[ESP_EXT] + cr[EXT_EXT], COL[EXT_EXT]),
        ("Nacidos fuera con nacionalidad española<br>(% de los nacidos fuera)", cr[ESP_EXT], cr[ESP_EXT] + cr[EXT_EXT], COL[ESP_EXT]),
    ]
    fig = make_subplots(rows=2, cols=2, horizontal_spacing=0.1, vertical_spacing=0.2,
                        subplot_titles=[p[0] for p in paneles])
    for k, (_, num, den, color) in enumerate(paneles):
        r, c = k // 2 + 1, k % 2 + 1
        p = (100 * num / den).to_list()
        fig.add_scatter(x=anios, y=p, mode="lines", line=dict(color=color, width=3), showlegend=False,
                        hovertext=[f"{_es(v, '.1f')} % · {_es(n, ',.0f')} de {_es(d, ',.0f')}"
                                   for v, n, d in zip(p, num.to_list(), den.to_list())],
                        hovertemplate="%{hovertext}", row=r, col=c)
        for i in (0, -1):
            fig.add_annotation(x=anios[i], y=p[i], text=f"{_es(p[i], '.1f')} %", showarrow=False,
                               xanchor="left" if i == 0 else "right", xshift=6 if i == 0 else -4,
                               yshift=-14 if (i == 0 or p[-1] < p[-2]) else 14, font=dict(color=color), row=r, col=c)
        fig.update_yaxes(rangemode="tozero", ticksuffix=" %", tickformat=".0f", row=r, col=c)
    fig.update_xaxes(dtick=4)
    fig.update_layout(height=720, margin=dict(l=60, r=40, t=80, b=55), hovermode="x unified")
    for a in fig.layout.annotations[:4]:
        a.font.size = 12
    add_source(fig, FUENTE)
    return fig

## Análisis

### El punto de partida: población total

**Qué se mira.** La población residente total a 1 de enero y su variación anual, para tener el contexto sobre el que se reparten las dos preguntas.

In [680]:
#| label: tbl-total
#| tbl-cap: "Población residente total a 1 de enero y variación respecto al año anterior"
txt_anio(nac.select("anio", "Total").join(variacion_anual(nac, ["Total"]), on="anio"))

año,Total,Δ Total,Δ% Total
"""2002""",41.035.271,null,null
"""2003""",41.827.836,792.565,"1,93"
"""2004""",42.547.454,719.618,"1,72"
"""2005""",43.296.335,748.881,"1,76"
"""2006""",44.009.969,713.634,"1,65"
"""2007""",44.784.659,774.690,"1,76"
"""2008""",45.668.938,884.279,"1,97"
"""2009""",46.239.271,570.333,"1,25"
"""2010""",46.486.621,247.350,"0,53"
"""2011""",46.667.175,180.554,"0,39"


In [681]:
#| label: fig-total
#| fig-cap: "Población residente en España a 1 de enero (barras: variación anual, con su porcentaje sobre el año anterior; línea: población total, eje derecho)"
fig_total(nac).show()

**Lectura.** La población residente pasa de 41,0 millones en 2002 a 49,1 millones en 2025 (+8,1 millones; +0,79 % anual compuesto). El ritmo no es constante (@fig-total, @tbl-total). Entre 2003 y 2008 el aumento supera las 700.000 personas cada año, con un máximo de +884.279 en 2008. Desde 2009 se frena (+570.333) y tras el máximo de 2012 (46,8 millones) la población cae hasta 46,4 millones en 2016, con un mínimo de −216.906 (−0,46 %) en 2014. Vuelve a crecer desde 2017 y en 2023 el aumento es de +598.634, el mayor desde 2008; en 2024 y 2025 es de +534.334 y +508.602.

### P1 · Por nacionalidad

#### Evolución y peso

**Qué se mira.** Cómo evolucionan por separado la población con nacionalidad española y la extranjera, y qué parte del total es extranjera. En el gráfico de la izquierda, 0 % es la población de cada grupo en 2002 y cada punto indica cuánto ha cambiado desde entonces (por ejemplo, +100 % es el doble).

In [682]:
#| label: tbl-peso-nac
#| tbl-cap: "Población por nacionalidad y peso de la población extranjera sobre el total"
txt_anio(nac.select("anio", "Española", "Extranjera", "Total",
                    (100 * pl.col("Extranjera") / pl.col("Total")).round(2).alias("% extranjera")))

año,Española,Extranjera,Total,% extranjera
"""2002""",39.297.299,1.737.972,41.035.271,"4,24"
"""2003""",39.465.808,2.362.027,41.827.836,"5,65"
"""2004""",39.646.666,2.900.788,42.547.454,"6,82"
"""2005""",39.866.131,3.430.204,43.296.335,"7,92"
"""2006""",40.079.053,3.930.916,44.009.969,"8,93"
"""2007""",40.335.225,4.449.434,44.784.659,"9,94"
"""2008""",40.582.643,5.086.295,45.668.938,"11,14"
"""2009""",40.852.612,5.386.659,46.239.271,"11,65"
"""2010""",41.084.042,5.402.579,46.486.621,"11,62"
"""2011""",41.354.734,5.312.440,46.667.175,"11,38"


In [683]:
#| label: fig-peso-nac
#| fig-cap: "Izquierda: población española y extranjera (millones). Centro: cambio acumulado desde 2002 (%). Derecha: peso de la población extranjera sobre el total"
fig_indice_peso(nac, "Española", "Extranjera", "la población extranjera").show()

**Lectura.** Las dos poblaciones siguen trayectorias muy distintas (@fig-peso-nac, @tbl-peso-nac). En los paneles izquierdo (niveles) y central (cambio acumulado), la población con nacionalidad española crece un 7,4 % entre 2002 y 2025 (de 39,30 a 42,22 millones) y se mueve entre 42,0 y 42,2 millones desde 2016. La extranjera crece un 298 %: casi se multiplica por cuatro (de 1,74 a 6,91 millones), pero no de forma continua. Sube hasta 5,40 millones en 2010, baja hasta 4,42 millones en 2016–2017 y vuelve a subir desde 2018. En el panel derecho, su peso sobre el total pasa del 4,2 % (2002) al 11,65 % (2009), baja hasta el 9,5 % (2016–2017) y alcanza el 14,1 % en 2025, el máximo de la serie.

#### Variación anual

**Qué se mira.** Cuánto suma o resta cada grupo cada año.

In [684]:
#| label: tbl-var-nac
#| tbl-cap: "Variación anual de la población por nacionalidad (Δ: personas; Δ%: porcentaje)"
txt_anio(variacion_anual(nac, ["Española", "Extranjera", "Total"]))

año,Δ Española,Δ Extranjera,Δ Total,Δ% Española,Δ% Extranjera,Δ% Total
"""2002""",null,null,null,null,null,null
"""2003""",168.509,624.055,792.565,"0,43","35,91","1,93"
"""2004""",180.858,538.761,719.618,"0,46","22,81","1,72"
"""2005""",219.465,529.416,748.881,"0,55","18,25","1,76"
"""2006""",212.922,500.712,713.634,"0,53","14,6","1,65"
"""2007""",256.172,518.518,774.690,"0,64","13,19","1,76"
"""2008""",247.418,636.861,884.279,"0,61","14,31","1,97"
"""2009""",269.969,300.364,570.333,"0,67","5,91","1,25"
"""2010""",231.430,15.920,247.350,"0,57","0,3","0,53"
"""2011""",270.692,-90.139,180.554,"0,66","-1,67","0,39"


In [685]:
#| label: fig-var-nac
#| fig-cap: "Variación anual de la población española y extranjera (barras apiladas) y del total (línea)"
fig_variacion_anual(nac, "Española", "Extranjera").show()

**Lectura.** El gráfico muestra cuánto aporta cada grupo cada año (@fig-var-nac, @tbl-var-nac); la línea es el total. Hasta 2008, la población extranjera aporta la mayor parte del aumento: entre +500.000 y +640.000 cada año, frente a +168.509 a +256.172 de la española. En 2009 su aportación cae a +300.364 y desde 2011 es negativa (−90.139), con el mínimo en 2014 (−388.232; −7,67 %). En esos años la población española sigue creciendo (hasta +270.692 en 2011) y compensa en parte la caída del total. La extranjera vuelve a crecer en 2018 (+159.669), alcanza +390.516 en 2020 y se modera en 2021 y 2022 (+161.424 y +106.344). En 2023 se dispara (+580.574; +10,5 %), con +412.662 en 2024 y +409.689 en 2025. La población española aporta menos de 125.000 al año desde 2018, y es negativa en 2018, 2021 y 2022.

#### Por periodos: cuánto crece cada grupo y quién explica el crecimiento

**Qué se mira.** Para cada periodo $[a, b]$ y cada grupo, con $P_t$ su población a 1 de enero del año $t$ y $T_t$ la del total:

| Medida | Definición | Se lee como |
|---|---|---|
| Media anual | $(P_b - P_a)/(b - a)$ | personas que gana (o pierde) el grupo cada año, de media |
| Tasa compuesta (%) | $100\,[(P_b/P_a)^{1/(b-a)} - 1]$ | % anual constante que llevaría de $P_a$ a $P_b$ |
| Contribución (%) | $100\,(P_b - P_a)/(T_b - T_a)$ | parte de la variación del total que aporta el grupo |

La contribución de todos los grupos suma 100 %. Si el total casi no cambia (o baja), el cociente se dispara o cambia de signo: en esos periodos conviene leer la media anual, no la contribución.

En el gráfico, cada tramo es la media anual del grupo y el rombo la del total. Al pasar el cursor: media anual · tasa anual · contribución.

In [686]:
#| label: tbl-periodos-nac
#| tbl-cap: "Variación por periodo y nacionalidad: media anual, tasa anual compuesta y contribución a la variación del total"
res_nac = resumen_periodos(nac, ["Española", "Extranjera"], "Total", PERIODOS_TOTAL)
res_nac

periodo,grupo,inicial,final,variación,media anual,tasa compuesta (%),contribución (%)
"""2002–2008""","""Española""",39.297.299,40.582.643,1.285.344,214.224,"0,54","27,7"
"""2002–2008""","""Extranjera""",1.737.972,5.086.295,3.348.323,558.054,"19,6","72,3"
"""2002–2008""","""Total""",41.035.271,45.668.938,4.633.667,772.278,"1,8","100,0"
"""2008–2014""","""Española""",40.582.643,41.819.392,1.236.749,206.125,"0,5","149,6"
"""2008–2014""","""Extranjera""",5.086.295,4.676.352,-409.943,-68.324,"-1,39","-49,6"
"""2008–2014""","""Total""",45.668.938,46.495.744,826.806,137.801,"0,3","100,0"
"""2014–2019""","""Española""",41.819.392,42.068.189,248.797,49.759,"0,12","58,8"
"""2014–2019""","""Extranjera""",4.676.352,4.850.762,174.410,34.882,"0,74","41,2"
"""2014–2019""","""Total""",46.495.744,46.918.951,423.207,84.641,"0,18","100,0"
"""2019–2025""","""Española""",42.068.189,42.216.326,148.137,24.690,"0,06","6,7"


In [687]:
#| label: fig-periodos-nac
#| fig-cap: "Variación media anual por periodo, según nacionalidad (cada tramo es la media anual del grupo; el rombo, la del total)"
fig_periodos(res_nac.filter(pl.col("periodo") != "2002–2025"), ["Española", "Extranjera"]).show()

**Lectura.** Cada tramo del gráfico es la variación media anual de un grupo; el rombo, la del total (@fig-periodos-nac, @tbl-periodos-nac).

- **2002–2008.** El total crece 772.278 personas al año (1,80 % anual). El 72,3 % del aumento es población extranjera (558.054 al año; +19,6 % anual), frente a 214.224 de la española (+0,54 %).
- **2008–2014.** El total crece solo 137.801 al año (0,30 %). La española suma 206.125 al año y la extranjera resta 68.324 (−1,39 % anual). Este periodo mezcla una subida de la extranjera hasta 2010 y una caída después. Como el total sube poco, las contribuciones se salen de 0–100 % (149,6 % la española; −49,6 % la extranjera): es más informativo leer las medias anuales.
- **2014–2019.** Crecimiento débil: 84.641 al año (0,18 %), con un 58,8 % de población española (49.759 al año) y un 41,2 % de extranjera (34.882).
- **2019–2025.** El total acelera a 368.224 al año (0,77 %) y el 93,3 % del aumento es población extranjera (343.535 al año; +6,08 % anual); la española aporta 24.690 (+0,06 %).
- **2002–2025.** La extranjera explica el 63,9 % del aumento (5,17 de 8,09 millones) y la española, el 36,1 % (2,92 millones).

#### La población extranjera por región de nacionalidad

**Qué se mira.** De qué regiones procede la población extranjera y cuáles explican su crecimiento. El gráfico tiene tres vistas (botones sobre el gráfico): «Personas», «Peso (%)» y «Variación media anual por periodo».

In [688]:
#| label: tbl-regiones-nac-niveles
#| tbl-cap: "Población extranjera por región de nacionalidad (personas)"
T_NAC = tablas_regiones(nac, REG_NAC, "Extranjera")
T_NAC["niveles"]

año,Europa,Sudamérica,África,Asia,América Central y Caribe,América del Norte,Oceanía,Apátridas,Extranjera
"""2002""",590.357,603.294,364.732,80.773,68.392,28.530,1.506,389,1.737.972
"""2008""",2.161.986,1.525.069,917.729,264.904,164.081,49.285,2.615,626,5.086.295
"""2014""",2.175.886,907.577,1.002.138,341.763,195.742,49.884,2.437,925,4.676.352
"""2019""",2.118.928,891.421,1.066.160,432.783,274.911,61.078,2.944,2.537,4.850.762
"""2025""",2.449.192,1.958.005,1.367.584,563.293,459.376,104.691,5.049,4.781,6.911.971


In [689]:
#| label: tbl-regiones-nac-peso
#| tbl-cap: "Peso de cada región sobre la población extranjera (%)"
T_NAC["peso"]

año,Europa,Sudamérica,África,Asia,América Central y Caribe,América del Norte,Oceanía,Apátridas
"""2002""","34,0","34,7","21,0","4,6","3,9","1,6","0,1","0,0"
"""2008""","42,5","30,0","18,0","5,2","3,2","1,0","0,1","0,0"
"""2014""","46,5","19,4","21,4","7,3","4,2","1,1","0,1","0,0"
"""2019""","43,7","18,4","22,0","8,9","5,7","1,3","0,1","0,1"
"""2025""","35,4","28,3","19,8","8,1","6,6","1,5","0,1","0,1"


In [690]:
#| label: tbl-regiones-nac-media
#| tbl-cap: "Variación media anual de la población extranjera por región de nacionalidad (personas por año)"
T_NAC["media"]

periodo,Europa,Sudamérica,África,Asia,América Central y Caribe,América del Norte,Oceanía,Apátridas
"""2002–2008""",261.938,153.629,92.166,30.688,15.948,3.459,185,40
"""2008–2014""",2.317,-102.915,14.068,12.810,5.277,100,-30,50
"""2014–2019""",-11.392,-3.231,12.804,18.204,15.834,2.239,101,322
"""2019–2025""",55.044,177.764,50.237,21.752,30.744,7.269,351,374
"""2002–2025""",80.819,58.900,43.602,20.979,16.999,3.311,154,191


In [691]:
#| label: tbl-regiones-nac-tasa
#| tbl-cap: "Tasa anual compuesta de la población extranjera por región de nacionalidad (% anual)"
T_NAC["tasa"]

periodo,Europa,Sudamérica,África,Asia,América Central y Caribe,América del Norte,Oceanía,Apátridas
"""2002–2008""","24,15","16,72","16,62","21,89","15,7","9,54","9,63","8,25"
"""2008–2014""","0,11","-8,29","1,48","4,34","2,98","0,2","-1,17","6,72"
"""2014–2019""","-0,53","-0,36","1,25","4,84","7,03","4,13","3,85","22,36"
"""2019–2025""","2,44","14,01","4,24","4,49","8,93","9,4","9,41","11,14"
"""2002–2025""","6,38","5,25","5,91","8,81","8,63","5,82","5,4","11,53"


In [692]:
#| label: tbl-regiones-nac-cambio
#| tbl-cap: "Variación total de cada región de nacionalidad entre el inicio y el fin de cada periodo (%)"
T_NAC["cambio"]

periodo,Europa,Sudamérica,África,Asia,América Central y Caribe,América del Norte,Oceanía,Apátridas
"""2002–2008""","266,2","152,8","151,6","228,0","139,9","72,7","73,6","60,9"
"""2008–2014""","0,6","-40,5","9,2","29,0","19,3","1,2","-6,8","47,8"
"""2014–2019""","-2,6","-1,8","6,4","26,6","40,4","22,4","20,8","174,3"
"""2019–2025""","15,6","119,6","28,3","30,2","67,1","71,4","71,5","88,5"
"""2002–2025""","314,9","224,6","275,0","597,4","571,7","267,0","235,3","1.129,0"


In [693]:
#| label: tbl-regiones-nac-contribucion
#| tbl-cap: "Contribución de cada región a la variación de la población extranjera (%)"
T_NAC["contribucion"]

periodo,Europa,Sudamérica,África,Asia,América Central y Caribe,América del Norte,Oceanía,Apátridas
"""2002–2008""","46,9","27,5","16,5","5,5","2,9","0,6","0,0","0,0"
"""2008–2014""","-3,4","150,6","-20,6","-18,7","-7,7","-0,1","0,0","-0,1"
"""2014–2019""","-32,7","-9,3","36,7","52,2","45,4","6,4","0,3","0,9"
"""2019–2025""","16,0","51,7","14,6","6,3","8,9","2,1","0,1","0,1"
"""2002–2025""","35,9","26,2","19,4","9,3","7,6","1,5","0,1","0,1"


In [694]:
#| label: fig-regiones-nac
#| fig-cap: "Población extranjera por región de nacionalidad. Vistas: personas, peso (%) y variación media anual por periodo"
fig_regiones(nac, REG_NAC, "Extranjera", "la población extranjera").show()

**Lectura.** Cada afirmación se apoya en una vista del gráfico (@fig-regiones-nac) y en una tabla.

- **Vista «Personas»** (@tbl-regiones-nac-niveles). En 2002, Europa (590.357) y Sudamérica (603.294) son las dos mayores regiones de nacionalidad. En 2008 Europa llega a 2,16 millones y Sudamérica a 1,53 millones. Sudamérica cae a 907.577 en 2014 y se recupera hasta 1.958.005 en 2025, año en que Europa tiene 2.449.192.
- **Vista «Peso (%)»** (@tbl-regiones-nac-peso). Sudamérica pasa del 34,7 % de la población extranjera (2002) al 19,4 % (2014) y al 28,3 % (2025). Europa sube del 34,0 % al 46,5 % (2014) y baja al 35,4 % (2025). África ronda el 18–22 % en todo el periodo (19,8 % en 2025). Asia pasa del 4,6 % al 8,1 % y América Central y Caribe, del 3,9 % al 6,6 %.
- **Vista «Variación media anual por periodo»** (@tbl-regiones-nac-media, @tbl-regiones-nac-contribucion). En 2002–2008, Europa aporta el 46,9 % del aumento de la población extranjera, Sudamérica el 27,5 % y África el 16,5 %. En 2008–2014, la extranjera cae (−68.324 al año) y Sudamérica resta 102.915 al año (150,6 % de la caída). En 2014–2019 Europa resta 11.392 al año y suben Asia (+18.204), América Central y Caribe (+15.834) y África (+12.804). En 2019–2025 Sudamérica explica el 51,7 % (+177.764 al año), Europa el 16,0 %, África el 14,6 % y América Central y Caribe el 8,9 %.

### P2 · Por país de origen

#### Evolución y peso

**Qué se mira.** Cómo evolucionan los nacidos en España y los nacidos en el extranjero, y qué parte del total nació fuera. Como en P1, el gráfico de la izquierda muestra el cambio acumulado respecto a 2002 (0 % = población de 2002).

In [695]:
#| label: tbl-peso-pais
#| tbl-cap: "Población por país de nacimiento y peso de los nacidos en el extranjero sobre el total"
txt_anio(pais.select("anio", "España", "Extranjero", "Total",
                     (100 * pl.col("Extranjero") / pl.col("Total")).round(2).alias("% nacidos en el extranjero")))

año,España,Extranjero,Total,% nacidos en el extranjero
"""2002""",38.701.173,2.334.098,41.035.271,"5,69"
"""2003""",38.843.742,2.984.094,41.827.836,"7,13"
"""2004""",38.999.785,3.547.669,42.547.454,"8,34"
"""2005""",39.189.109,4.107.226,43.296.335,"9,49"
"""2006""",39.371.997,4.637.971,44.009.969,"10,54"
"""2007""",39.584.597,5.200.061,44.784.659,"11,61"
"""2008""",39.790.019,5.878.919,45.668.938,"12,87"
"""2009""",40.013.758,6.225.513,46.239.271,"13,46"
"""2010""",40.206.557,6.280.064,46.486.621,"13,51"
"""2011""",40.384.966,6.282.208,46.667.175,"13,46"


In [696]:
#| label: fig-peso-pais
#| fig-cap: "Izquierda: nacidos en España y en el extranjero (millones). Centro: cambio acumulado desde 2002 (%). Derecha: peso de los nacidos en el extranjero sobre el total"
fig_indice_peso(pais, "España", "Extranjero", "los nacidos en el extranjero").show()

**Lectura.** El gráfico y la tabla (@fig-peso-pais, @tbl-peso-pais) muestran una divergencia aún más marcada que por nacionalidad. Los nacidos en el extranjero pasan de 2,33 millones (5,7 %) en 2002 a 9,46 millones (19,3 %) en 2025: el aumento acumulado es del 305 %. Los nacidos en España crecen un 2,5 % (de 38,70 a 39,66 millones) pero con un máximo en 2013 (40,55 millones) y una caída de 882.928 personas desde entonces. Los nacidos en el extranjero se estancan en torno a 6,3 millones entre 2010 y 2012, bajan hasta 5,88 millones en 2015 (−6,5 % desde 2012) y crecen todos los años desde 2016. Su peso apenas retrocede en la crisis, del 13,5 % (2010) al 12,7 % (2015), mientras que el de la población extranjera por nacionalidad baja del 11,6 % (2009) al 9,5 % (2017); después sube hasta el 19,3 %.

#### Por periodos

**Qué se mira.** Lo mismo que en P1, con nacidos en España y nacidos en el extranjero.

In [697]:
#| label: tbl-periodos-pais
#| tbl-cap: "Variación por periodo y país de nacimiento: media anual, tasa anual compuesta y contribución a la variación del total"
res_pais = resumen_periodos(pais, ["España", "Extranjero"], "Total", PERIODOS_TOTAL)
res_pais

periodo,grupo,inicial,final,variación,media anual,tasa compuesta (%),contribución (%)
"""2002–2008""","""España""",38.701.173,39.790.019,1.088.846,181.474,"0,46","23,5"
"""2002–2008""","""Extranjero""",2.334.098,5.878.919,3.544.821,590.804,"16,64","76,5"
"""2002–2008""","""Total""",41.035.271,45.668.938,4.633.667,772.278,"1,8","100,0"
"""2008–2014""","""España""",39.790.019,40.542.974,752.955,125.492,"0,31","91,1"
"""2008–2014""","""Extranjero""",5.878.919,5.952.770,73.851,12.308,"0,21","8,9"
"""2008–2014""","""Total""",45.668.938,46.495.744,826.806,137.801,"0,3","100,0"
"""2014–2019""","""España""",40.542.974,40.369.642,-173.332,-34.666,"-0,09","-41,0"
"""2014–2019""","""Extranjero""",5.952.770,6.549.309,596.539,119.308,"1,93","141,0"
"""2014–2019""","""Total""",46.495.744,46.918.951,423.207,84.641,"0,18","100,0"
"""2019–2025""","""España""",40.369.642,39.664.087,-705.555,-117.592,"-0,29","-31,9"


In [698]:
#| label: fig-periodos-pais
#| fig-cap: "Variación media anual por periodo, según país de nacimiento (cada tramo es la media anual del grupo; el rombo, la del total)"
fig_periodos(res_pais.filter(pl.col("periodo") != "2002–2025"), ["España", "Extranjero"]).show()

**Lectura.** Igual que en P1 (@fig-periodos-pais, @tbl-periodos-pais):

- **2002–2008.** De los 772.278 al año del total, 590.804 son nacidos en el extranjero (76,5 %) y 181.474, nacidos en España.
- **2008–2014.** El total crece 137.801 al año y casi todo (125.492; 91,1 %) son nacidos en España. Los nacidos en el extranjero apenas aportan 12.308 al año (+0,21 % anual).
- **2014–2019.** Los nacidos en España bajan 34.666 al año (−0,09 %) y los nacidos en el extranjero suben 119.308 (+1,93 % anual); la contribución es de −41,0 % y 141,0 %.
- **2019–2025.** Los nacidos en España bajan 117.592 al año (−0,29 %) y los nacidos en el extranjero suben 485.817 (+6,33 % anual); contribuciones de −31,9 % y 131,9 %.
- **2002–2025.** Los nacidos en el extranjero explican el 88,1 % del aumento (7,13 de 8,09 millones), frente al 63,9 % que explica la población extranjera por nacionalidad en P1.

#### Los nacidos en el extranjero por región de nacimiento

**Qué se mira.** De qué regiones proceden los nacidos en el extranjero y cuáles explican su crecimiento. Mismas tres vistas que en P1.

In [699]:
#| label: tbl-regiones-pais-niveles
#| tbl-cap: "Población nacida en el extranjero por región de nacimiento (personas)"
T_PAIS = tablas_regiones(pais, REG_PAIS, "Extranjero")
T_PAIS["niveles"]

año,Europa,Sudamérica,África,Asia,América Central y Caribe,América del Norte,Oceanía,Extranjero
"""2002""",844.816,780.432,442.408,98.034,115.042,48.736,4.631,2.334.098
"""2008""",2.300.335,1.981.303,961.656,290.617,258.754,79.596,6.659,5.878.919
"""2014""",2.276.765,1.834.604,1.017.292,361.951,364.165,91.135,6.858,5.952.770
"""2019""",2.165.240,2.147.783,1.157.240,452.241,508.723,110.143,7.939,6.549.309
"""2025""",2.417.433,3.739.859,1.647.610,616.222,858.231,174.180,10.675,9.464.210


In [700]:
#| label: tbl-regiones-pais-peso
#| tbl-cap: "Peso de cada región sobre los nacidos en el extranjero (%)"
T_PAIS["peso"]

año,Europa,Sudamérica,África,Asia,América Central y Caribe,América del Norte,Oceanía
"""2002""","36,2","33,4","19,0","4,2","4,9","2,1","0,2"
"""2008""","39,1","33,7","16,4","4,9","4,4","1,4","0,1"
"""2014""","38,2","30,8","17,1","6,1","6,1","1,5","0,1"
"""2019""","33,1","32,8","17,7","6,9","7,8","1,7","0,1"
"""2025""","25,5","39,5","17,4","6,5","9,1","1,8","0,1"


In [701]:
#| label: tbl-regiones-pais-media
#| tbl-cap: "Variación media anual de los nacidos en el extranjero por región de nacimiento (personas por año)"
T_PAIS["media"]

periodo,Europa,Sudamérica,África,Asia,América Central y Caribe,América del Norte,Oceanía
"""2002–2008""",242.586,200.145,86.541,32.097,23.952,5.143,338
"""2008–2014""",-3.928,-24.450,9.273,11.889,17.568,1.923,33
"""2014–2019""",-22.305,62.636,27.990,18.058,28.912,3.802,216
"""2019–2025""",42.032,265.346,81.728,27.330,58.251,10.673,456
"""2002–2025""",68.375,128.671,52.400,22.530,32.313,5.454,263


In [702]:
#| label: tbl-regiones-pais-tasa
#| tbl-cap: "Tasa anual compuesta de los nacidos en el extranjero por región de nacimiento (% anual)"
T_PAIS["tasa"]

periodo,Europa,Sudamérica,África,Asia,América Central y Caribe,América del Norte,Oceanía
"""2002–2008""","18,17","16,8","13,81","19,86","14,46","8,52","6,24"
"""2008–2014""","-0,17","-1,27","0,94","3,73","5,86","2,28","0,49"
"""2014–2019""","-1,0","3,2","2,61","4,55","6,91","3,86","2,97"
"""2019–2025""","1,85","9,68","6,06","5,29","9,11","7,94","5,06"
"""2002–2025""","4,68","7,05","5,88","8,32","9,13","5,69","3,7"


In [703]:
#| label: tbl-regiones-pais-cambio
#| tbl-cap: "Variación total de cada región de nacimiento entre el inicio y el fin de cada periodo (%)"
T_PAIS["cambio"]

periodo,Europa,Sudamérica,África,Asia,América Central y Caribe,América del Norte,Oceanía
"""2002–2008""","172,3","153,9","117,4","196,4","124,9","63,3","43,8"
"""2008–2014""","-1,0","-7,4","5,8","24,5","40,7","14,5","3,0"
"""2014–2019""","-4,9","17,1","13,8","24,9","39,7","20,9","15,8"
"""2019–2025""","11,6","74,1","42,4","36,3","68,7","58,1","34,5"
"""2002–2025""","186,1","379,2","272,4","528,6","646,0","257,4","130,5"


In [704]:
#| label: tbl-regiones-pais-contribucion
#| tbl-cap: "Contribución de cada región a la variación de los nacidos en el extranjero (%)"
T_PAIS["contribucion"]

periodo,Europa,Sudamérica,África,Asia,América Central y Caribe,América del Norte,Oceanía
"""2002–2008""","41,1","33,9","14,6","5,4","4,1","0,9","0,1"
"""2008–2014""","-31,9","-198,6","75,3","96,6","142,7","15,6","0,3"
"""2014–2019""","-18,7","52,5","23,5","15,1","24,2","3,2","0,2"
"""2019–2025""","8,7","54,6","16,8","5,6","12,0","2,2","0,1"
"""2002–2025""","22,1","41,5","16,9","7,3","10,4","1,8","0,1"


In [705]:
#| label: fig-regiones-pais
#| fig-cap: "Población nacida en el extranjero por región de nacimiento. Vistas: personas, peso (%) y variación media anual por periodo"
fig_regiones(pais, REG_PAIS, "Extranjero", "los nacidos en el extranjero").show()

**Lectura.** Cada afirmación se apoya en una vista del gráfico (@fig-regiones-pais) y en una tabla.

- **Vista «Personas»** (@tbl-regiones-pais-niveles). En 2002, los nacidos en Europa (844.816) y en Sudamérica (780.432) son las dos mayores regiones. En 2025, Sudamérica (3.739.859) supera en un 55 % a Europa (2.417.433), seguida de África (1.647.610), América Central y Caribe (858.231) y Asia (616.222).
- **Vista «Peso (%)»** (@tbl-regiones-pais-peso). Sudamérica pasa del 33,4 % al 39,5 % de los nacidos en el extranjero y Europa baja del 36,2 % al 25,5 %. América Central y Caribe sube del 4,9 % al 9,1 %.
- **Vista «Variación media anual por periodo»** (@tbl-regiones-pais-media, @tbl-regiones-pais-contribucion). En 2002–2008 Europa aporta el 41,1 % del aumento y Sudamérica el 33,9 %. En 2019–2025 Sudamérica aporta 265.346 al año (54,6 %), África 81.728 (16,8 %) y América Central y Caribe 58.251 (12,0 %), mientras que Europa aporta 42.032 (8,7 %). En 2008–2014 el total de nacidos en el extranjero casi no cambia (+12.308 al año), por lo que las contribuciones en % son extremas (Sudamérica −198,6 %; América Central y Caribe 142,7 %): conviene leer ese periodo con las medias anuales.
- **Comparación con la nacionalidad.** En 2025, Sudamérica tiene 3.739.859 nacidos frente a 1.958.005 con nacionalidad sudamericana, mientras que en Europa las dos cifras son parecidas (2.417.433 y 2.449.192). La diferencia entre ambas definiciones no es igual en todas las regiones; el cruce siguiente la cuantifica en conjunto.

### Nacionalidad frente a lugar de nacimiento

**Qué se mira.** «Español» y «extranjero» pueden definirse de dos maneras, y no coinciden:

| Definición | Es español… | Es extranjero… |
|---|---|---|
| **Por nacionalidad** (P1) | todo residente con nacionalidad española, haya nacido o no en España | todo residente con nacionalidad extranjera, haya nacido donde haya nacido |
| **Por lugar de nacimiento** (P2) | todo residente nacido en España, tenga o no nacionalidad española | todo residente nacido en el extranjero, tenga la nacionalidad que tenga |

Al cruzar las dos variables resultan cuatro grupos: nacionalidad española nacida en España, nacionalidad española nacida en el extranjero, nacionalidad extranjera nacida en España y nacionalidad extranjera nacida en el extranjero. Las dos definiciones de «extranjero» coinciden en el último grupo y difieren en los otros dos: los residentes con nacionalidad española nacidos fuera cuentan como extranjeros solo por lugar de nacimiento, y los residentes con nacionalidad extranjera nacidos en España solo por nacionalidad.

El análisis tiene cuatro pasos: cuántas personas hay en cada grupo, cuánto difieren las definiciones, qué explica la diferencia y cómo ha cambiado cada grupo.

#### 1. Los cuatro grupos: cuántas personas y qué peso

In [706]:
#| label: tbl-cruce
#| tbl-cap: "Población según nacionalidad y lugar de nacimiento (personas)"
txt_anio(cruce.select("anio", ESP_ESP, ESP_EXT, EXT_ESP, EXT_EXT, "Total"))

año,Nacionalidad española · nacidos en España,Nacionalidad española · nacidos en el extranjero,Nacionalidad extranjera · nacidos en España,Nacionalidad extranjera · nacidos en el extranjero,Total
"""2002""",38.621.928,675.371,79.245,1.658.727,41.035.271
"""2003""",38.744.787,721.021,98.954,2.263.073,41.827.835
"""2004""",38.877.064,769.603,122.721,2.778.066,42.547.454
"""2005""",39.040.468,825.663,148.641,3.281.563,43.296.335
"""2006""",39.194.375,884.677,177.622,3.753.294,44.009.968
"""2007""",39.373.663,961.562,210.934,4.238.500,44.784.659
"""2008""",39.537.144,1.045.500,252.875,4.833.420,45.668.939
"""2009""",39.716.623,1.135.988,297.134,5.089.525,46.239.270
"""2010""",39.865.937,1.218.105,340.620,5.061.959,46.486.621
"""2011""",40.011.670,1.343.065,373.297,4.939.144,46.667.176


In [707]:
#| label: tbl-cruce-peso
#| tbl-cap: "Peso de cada grupo sobre la población total (%)"
txt_anio(cruce.filter(pl.col("anio").is_in(ANIOS_CLAVE)).select(
    "anio", *[(100 * pl.col(g) / pl.col("Total")).round(1).alias(g) for g in (ESP_ESP, ESP_EXT, EXT_ESP, EXT_EXT)]))

año,Nacionalidad española · nacidos en España,Nacionalidad española · nacidos en el extranjero,Nacionalidad extranjera · nacidos en España,Nacionalidad extranjera · nacidos en el extranjero
"""2002""","94,1","1,6","0,2","4,0"
"""2008""","86,6","2,3","0,6","10,6"
"""2014""","86,3","3,7","0,9","9,1"
"""2019""","84,9","4,7","1,1","9,2"
"""2025""","79,7","6,2","1,0","13,1"


In [708]:
#| label: tbl-cruce-peso-cambio
#| tbl-cap: "Cambio del peso de cada grupo entre el inicio y el fin de cada periodo (puntos porcentuales)"
GR4 = [ESP_ESP, ESP_EXT, EXT_ESP, EXT_EXT]
(cruce.filter(pl.col("anio").is_in(ANIOS_CLAVE))
      .select("anio", *[(100 * pl.col(g) / pl.col("Total")).alias(g) for g in GR4])
      .with_columns(*[pl.col(g).diff().round(1) for g in GR4],
                    periodo=pl.format("{}–{}", pl.col("anio").shift(1), pl.col("anio")))
      .drop_nulls("periodo")
      .select("periodo", *GR4))

periodo,Nacionalidad española · nacidos en España,Nacionalidad española · nacidos en el extranjero,Nacionalidad extranjera · nacidos en España,Nacionalidad extranjera · nacidos en el extranjero
"""2002–2008""","-7,5","0,6","0,4","6,5"
"""2008–2014""","-0,3","1,4","0,4","-1,5"
"""2014–2019""","-1,3","1,0","0,2","0,1"
"""2019–2025""","-5,2","1,5","-0,1","3,8"


In [709]:
#| label: fig-cruce-pesos
#| fig-cap: "Peso sobre la población total de los tres grupos distintos de «nacionalidad española · nacidos en España» (%; el resto, hasta 100 %, es ese grupo)"
fig_cruce_pesos(cruce).show()

**Lectura.** La tabla de niveles (@tbl-cruce), la de pesos (@tbl-cruce-peso) y las barras (@fig-cruce-pesos) muestran cómo se reparte la población entre los cuatro grupos.

- Los residentes con nacionalidad española nacidos en España pasan del 94,1 % de la población (2002) al 79,7 % (2025). En personas, sin embargo, suben de 38,62 a 39,16 millones (+543.016): alcanzan su máximo en 2012 (40,13 millones) y bajan desde 2013.
- En puntos porcentuales (@tbl-cruce-peso-cambio), los nacidos en España con nacionalidad española pierden 7,5 puntos en 2002–2008 y 5,2 en 2019–2025, y los de nacionalidad extranjera nacidos fuera ganan 6,5 y 3,8 puntos; en 2008–2014 estos últimos pierden 1,5 puntos. Los de nacionalidad española nacidos fuera ganan peso en los cuatro periodos (de +0,6 a +1,5 puntos).
- Los tres grupos restantes (la altura de las barras) suman el 5,9 % en 2002 y el 20,3 % en 2025.
- Nacionalidad extranjera, nacidos en el extranjero: 4,0 % (2002), 10,6 % (2008), 9,1 % (2014), 9,2 % (2019) y 13,1 % (2025).
- Nacionalidad española, nacidos en el extranjero: del 1,6 % al 6,2 %. Su peso sube todos los años (de 675.371 a 3.051.382 personas).
- Nacionalidad extranjera, nacidos en España: 0,2 % (2002), 1,1 % (2019) y 1,0 % (2025).

#### 2. Dos definiciones de «extranjero»: cuánto difieren

In [710]:
#| label: tbl-cruce-definiciones
#| tbl-cap: "Dos definiciones de «extranjero»: residentes con nacionalidad extranjera y residentes nacidos en el extranjero (personas y % de la población total)"
d = cruce.filter(pl.col("anio").is_in(ANIOS_CLAVE))
txt_anio(d.select(
    "anio",
    (pl.col(EXT_ESP) + pl.col(EXT_EXT)).alias("Con nacionalidad extranjera"),
    (pl.col(ESP_EXT) + pl.col(EXT_EXT)).alias("Nacidos en el extranjero"),
    (pl.col(ESP_EXT) - pl.col(EXT_ESP)).alias("Diferencia (personas)"),
    (100 * (pl.col(EXT_ESP) + pl.col(EXT_EXT)) / pl.col("Total")).round(1).alias("% con nacionalidad extranjera"),
    (100 * (pl.col(ESP_EXT) + pl.col(EXT_EXT)) / pl.col("Total")).round(1).alias("% nacidos en el extranjero"),
    (100 * (pl.col(ESP_EXT) - pl.col(EXT_ESP)) / pl.col("Total")).round(1).alias("Diferencia (p. p.)"),
))

año,Con nacionalidad extranjera,Nacidos en el extranjero,Diferencia (personas),% con nacionalidad extranjera,% nacidos en el extranjero,Diferencia (p. p.)
"""2002""",1.737.972,2.334.098,596.126,"4,2","5,7","1,5"
"""2008""",5.086.295,5.878.920,792.625,"11,1","12,9","1,7"
"""2014""",4.676.352,5.952.770,1.276.418,"10,1","12,8","2,7"
"""2019""",4.850.762,6.549.309,1.698.547,"10,3","14,0","3,6"
"""2025""",6.911.971,9.464.210,2.552.239,"14,1","19,3","5,2"


In [711]:
#| label: fig-cruce-definiciones
#| fig-cap: "Población «extranjera» según se defina por nacionalidad o por lugar de nacimiento. Izquierda: millones de personas. Derecha: peso sobre la población total (%)"
fig_definiciones(cruce).show()

**Lectura.** La tabla (@tbl-cruce-definiciones) y la figura (@fig-cruce-definiciones) comparan la población «extranjera» según cada definición: residentes con nacionalidad extranjera (nazcan donde nazcan) frente a residentes nacidos en el extranjero (tengan la nacionalidad que tengan).

- La definición por lugar de nacimiento da siempre una cifra mayor: en 2002, 2,33 millones frente a 1,74 con nacionalidad extranjera (5,7 % frente a 4,2 % de la población); en 2025, 9,46 frente a 6,91 millones (19,3 % frente a 14,1 %).
- La diferencia es de 596.126 personas en 2002 (1,5 puntos porcentuales), 792.625 en 2008 (1,7), 1.276.418 en 2014 (2,7), 1.698.547 en 2019 (3,6) y 2.552.239 en 2025 (5,2). Crece todos los años de la serie.
- En el panel derecho se ve que la distancia entre ambas líneas se abre sobre todo desde 2008.

#### 3. Qué explica la diferencia

La diferencia entre ambas definiciones es igual a los residentes con nacionalidad española nacidos en el extranjero menos los residentes con nacionalidad extranjera nacidos en España. Se miran tres proporciones que la resumen.

In [712]:
#| label: tbl-cruce-explica
#| tbl-cap: "Los dos grupos que explican la diferencia: personas y % dentro de su grupo de nacionalidad"
txt_anio(cruce.filter(pl.col("anio").is_in(ANIOS_CLAVE)).select(
    "anio",
    pl.col(ESP_EXT).alias("Nacionalidad española, nacidos fuera"),
    (100 * pl.col(ESP_EXT) / (pl.col(ESP_ESP) + pl.col(ESP_EXT))).round(1).alias("% de los de nacionalidad española"),
    pl.col(EXT_ESP).alias("Nacionalidad extranjera, nacidos en España"),
    (100 * pl.col(EXT_ESP) / (pl.col(EXT_ESP) + pl.col(EXT_EXT))).round(1).alias("% de los de nacionalidad extranjera"),
    (100 * pl.col(EXT_EXT) / (pl.col(ESP_EXT) + pl.col(EXT_EXT))).round(1).alias("% de los nacidos fuera con nacionalidad extranjera"),
    (100 * pl.col(ESP_EXT) / (pl.col(ESP_EXT) + pl.col(EXT_EXT))).round(1).alias("% de los nacidos fuera con nacionalidad española"),
))

año,"Nacionalidad española, nacidos fuera",% de los de nacionalidad española,"Nacionalidad extranjera, nacidos en España",% de los de nacionalidad extranjera,% de los nacidos fuera con nacionalidad extranjera,% de los nacidos fuera con nacionalidad española
"""2002""",675.371,"1,7",79.245,"4,6","71,1","28,9"
"""2008""",1.045.500,"2,6",252.875,"5,0","82,2","17,8"
"""2014""",1.709.994,"4,1",433.576,"9,3","71,3","28,7"
"""2019""",2.215.131,"5,3",516.584,"10,6","66,2","33,8"
"""2025""",3.051.382,"7,2",499.143,"7,2","67,8","32,2"


In [713]:
#| label: fig-cruce-explica
#| fig-cap: "Arriba izquierda: residentes con nacionalidad española nacidos fuera (% de los de nacionalidad española). Arriba derecha: residentes con nacionalidad extranjera nacidos en España (% de los de nacionalidad extranjera). Abajo izquierda: nacidos fuera con nacionalidad extranjera (% de los nacidos fuera). Abajo derecha: nacidos fuera con nacionalidad española (% de los nacidos fuera)"
fig_explica(cruce).show()

**Lectura.** La diferencia entre definiciones es igual a los residentes con nacionalidad española nacidos fuera menos los residentes con nacionalidad extranjera nacidos en España (@tbl-cruce-explica, @fig-cruce-explica).

- **Nacionalidad española y nacidos fuera (arriba izquierda).** Son el 1,7 % de los residentes con nacionalidad española en 2002 y el 7,2 % en 2025 (3.051.382 personas). La proporción sube todos los años.
- **Nacionalidad extranjera y nacidos en España (arriba derecha).** Son el 4,6 % de los residentes con nacionalidad extranjera en 2002, el 5,0 % en 2008 y el 9,3 % en 2014. Alcanzan el máximo en 2018 (10,8 %) y bajan hasta el 7,2 % en 2025 (499.143 personas).
- **Nacidos fuera con nacionalidad extranjera (abajo izquierda).** Son el 71,1 % de los nacidos fuera en 2002, el máximo en 2008 (82,2 %), el mínimo en 2017 (65,6 %) y el 67,8 % en 2025. Su complementario (abajo derecha) son los nacidos fuera con nacionalidad española: 28,9 % en 2002, 17,8 % en 2008, el máximo en 2017 (34,4 %) y 32,2 % en 2025, es decir, uno de cada tres.
- Entre 2002 y 2025 la diferencia aumenta en 1.956.113 personas: los residentes con nacionalidad española nacidos fuera suben 2.376.011 y los de nacionalidad extranjera nacidos en España, 419.898. El aumento de la diferencia se debe sobre todo al primer grupo.

#### 4. Cómo ha cambiado cada grupo por periodos

In [714]:
#| label: tbl-cruce-periodos
#| tbl-cap: "Variación por periodo de los cuatro grupos: media anual, tasa anual compuesta y contribución a la variación del total"
res_cruce = resumen_periodos(cruce, GR4, "Total", PERIODOS_TOTAL)
res_cruce

periodo,grupo,inicial,final,variación,media anual,tasa compuesta (%),contribución (%)
"""2002–2008""","""Nacionalidad española · nacido…",38.621.928,39.537.144,915.216,152.536,"0,39","19,8"
"""2002–2008""","""Nacionalidad española · nacido…",675.371,1.045.500,370.129,61.688,"7,55","8,0"
"""2002–2008""","""Nacionalidad extranjera · naci…",79.245,252.875,173.630,28.938,"21,34","3,7"
"""2002–2008""","""Nacionalidad extranjera · naci…",1.658.727,4.833.420,3.174.693,529.116,"19,51","68,5"
"""2002–2008""","""Total""",41.035.271,45.668.939,4.633.668,772.278,"1,8","100,0"
"""2008–2014""","""Nacionalidad española · nacido…",39.537.144,40.109.398,572.254,95.376,"0,24","69,2"
"""2008–2014""","""Nacionalidad española · nacido…",1.045.500,1.709.994,664.494,110.749,"8,55","80,4"
"""2008–2014""","""Nacionalidad extranjera · naci…",252.875,433.576,180.701,30.117,"9,4","21,9"
"""2008–2014""","""Nacionalidad extranjera · naci…",4.833.420,4.242.776,-590.644,-98.441,"-2,15","-71,4"
"""2008–2014""","""Total""",45.668.939,46.495.744,826.805,137.801,"0,3","100,0"


In [715]:
#| label: tbl-cruce-cambio
#| tbl-cap: "Variación total de cada grupo entre el inicio y el fin de cada periodo (%)"
(res_cruce.filter(pl.col("grupo") != "Total")
          .with_columns((100 * (pl.col("final") / pl.col("inicial") - 1)).round(1).alias("variación (%)"))
          .pivot(on="grupo", index="periodo", values="variación (%)")
          .select("periodo", *GR4))

periodo,Nacionalidad española · nacidos en España,Nacionalidad española · nacidos en el extranjero,Nacionalidad extranjera · nacidos en España,Nacionalidad extranjera · nacidos en el extranjero
"""2002–2008""","2,4","54,8","219,1","191,4"
"""2008–2014""","1,4","63,6","71,5","-12,2"
"""2014–2019""","-0,6","29,5","19,1","2,2"
"""2019–2025""","-1,7","37,8","-3,4","48,0"
"""2002–2025""","1,4","351,8","529,9","286,6"


In [716]:
#| label: fig-cruce-periodos
#| fig-cap: "Variación media anual por periodo de los cuatro grupos (cada tramo es la media anual del grupo; el rombo, la del total)"
fig_periodos(res_cruce.filter(pl.col("periodo") != "2002–2025"), GR4).show()

**Lectura.** La media anual, la tasa compuesta y la contribución (@tbl-cruce-periodos), el cambio total en % (@tbl-cruce-cambio) y las barras (@fig-cruce-periodos) describen el cambio de cada grupo.

- **Nacionalidad extranjera, nacidos en el extranjero.** +191,4 % en 2002–2008 (media anual de 529.116), −12,2 % en 2008–2014 (−98.441 al año), +2,2 % en 2014–2019 y +48,0 % en 2019–2025 (346.442 al año, el 94,1 % de la variación del total).
- **Nacionalidad española, nacidos en el extranjero.** Crecen en todos los periodos: +54,8 %, +63,6 %, +29,5 % y +37,8 %. En 2019–2025 suman 139.375 al año, el 37,9 % de la variación del total.
- **Nacionalidad extranjera, nacidos en España.** +219,1 % en 2002–2008, +71,5 % en 2008–2014, +19,1 % en 2014–2019 y −3,4 % en 2019–2025 (−2.907 al año).
- **Nacionalidad española, nacidos en España.** +2,4 %, +1,4 %, −0,6 % y −1,7 %. En 2019–2025 pierden 114.686 al año (contribución del −31,1 %).
- Entre 2002 y 2025: nacionalidad española nacidos fuera +351,8 %, nacionalidad extranjera nacidos en España +529,9 %, nacionalidad extranjera nacidos fuera +286,6 % y nacionalidad española nacidos en España +1,4 %.
- Cuando el total cambia poco (2008–2014: +826.805; 2014–2019: +423.207) las contribuciones se salen de 0–100 % (p. ej., 119,4 % y −60,6 % en 2014–2019); en esos periodos conviene leer la media anual y el cambio en %.

### Robustez: ¿dependen los resultados de los cortes de periodo?

**Qué se mira.** Se repite el análisis de P1 con bloques de 5 años (el último, de 3).

In [717]:
#| label: tbl-robustez
#| tbl-cap: "Variación por bloques de 5 años y nacionalidad: media anual, tasa anual compuesta y contribución"
res_5 = resumen_periodos(nac, ["Española", "Extranjera"], "Total", BLOQUES_5)
res_5

periodo,grupo,inicial,final,variación,media anual,tasa compuesta (%),contribución (%)
"""2002–2007""","""Española""",39.297.299,40.335.225,1.037.926,207.585,"0,52","27,7"
"""2002–2007""","""Extranjera""",1.737.972,4.449.434,2.711.462,542.292,"20,68","72,3"
"""2002–2007""","""Total""",41.035.271,44.784.659,3.749.388,749.878,"1,76","100,0"
"""2007–2012""","""Española""",40.335.225,41.582.186,1.246.961,249.392,"0,61","61,3"
"""2007–2012""","""Extranjera""",4.449.434,5.236.030,786.596,157.319,"3,31","38,7"
"""2007–2012""","""Total""",44.784.659,46.818.216,2.033.557,406.711,"0,89","100,0"
"""2012–2017""","""Española""",41.582.186,42.079.740,497.554,99.511,"0,24","-155,1"
"""2012–2017""","""Extranjera""",5.236.030,4.417.653,-818.377,-163.675,"-3,34","255,1"
"""2012–2017""","""Total""",46.818.216,46.497.393,-320.823,-64.165,"-0,14","100,0"
"""2017–2022""","""Española""",42.079.740,41.977.681,-102.059,-20.412,"-0,05","-10,3"


In [718]:
#| label: fig-robustez
#| fig-cap: "Variación media anual por bloques de 5 años, según nacionalidad"
fig_periodos(res_5, ["Española", "Extranjera"]).show()

**Lectura.** Con bloques de 5 años (@tbl-robustez, @fig-robustez) la historia de fondo se mantiene, aunque los números cambian:

- **2002–2007.** La extranjera explica el 72,3 % del aumento (542.292 al año; +20,68 % anual) y la española aporta 207.585 al año.
- **2007–2012.** La extranjera aporta 157.319 al año (38,7 %; +3,31 % anual) y la española 249.392 (61,3 %).
- **2012–2017.** El total cae (−64.165 al año). La extranjera resta 163.675 al año (−3,34 % anual) y la española suma 99.511. Las contribuciones (255,1 % y −155,1 %) se leen sobre una caída del total.
- **2017–2022.** La extranjera suma 218.279 al año (110,3 %; +4,51 % anual) y la española resta 20.412 (−10,3 %).
- **2022–2025.** La extranjera suma 467.642 al año (85,5 %; +7,86 % anual) y la española 79.548 (14,5 %).

El patrón no depende de los cortes: la población extranjera crece hasta 2008–2010, retrocede hasta 2016–2017 y repunta después, con más fuerza desde 2023. Lo que sí depende de los cortes son las cifras concretas de cada periodo, sobre todo cuando el total varía poco o cae.

::: {.callout-note collapse="true" title="Detalle técnico"}
**Definiciones y fórmulas.** Sea $P_{g,t}$ la población del grupo $g$ a 1 de enero del año $t$ y $P_t$ la del total.

- Variación anual: $\Delta P_{g,t} = P_{g,t} - P_{g,t-1}$, atribuida al año $t$; en %, $100\,\Delta P_{g,t} / P_{g,t-1}$.
- Media anual en el periodo $[t_0, t_1]$: $(P_{g,t_1} - P_{g,t_0}) / (t_1 - t_0)$.
- Tasa anual compuesta: $\left(P_{g,t_1} / P_{g,t_0}\right)^{1/(t_1 - t_0)} - 1$.
- Peso: $100\,P_{g,t} / P_t$.
- Contribución al crecimiento: $(P_{g,t_1} - P_{g,t_0}) / (P_{t_1} - P_{t_0})$; puede ser negativa o superar el 100 %.

**Tratamiento de los datos.** Solo población a 1 de enero. Las clasificaciones de Europa (UE28 hasta 2020 y UE27 desde 2020) se unifican en una sola región. En 2020 no se duplica ninguna región: la suma de regiones coincide con la población extranjera (diferencia máxima de 2 personas en toda la serie).

**Comprobaciones de calidad.** Española + Extranjera = Total; regiones de nacionalidad (con apátridas) = Extranjera; España + Extranjero = Total; regiones de nacimiento = Extranjero. Diferencias máximas observadas: 1, 2, 1 y 1 personas (redondeo del INE); sin valores nulos tras unir Europa. Código en `poblacion/src/process/`.

**Cómo leer la contribución.** Si la variación del total es pequeña o negativa, la contribución puede salirse de 0–100 % y su signo se invierte: con el total a la baja, un porcentaje positivo indica que el grupo también cae. En esos periodos (2008–2014 en P1 y P2; 2012–2017 en la robustez) es más informativo leer las medias anuales.
:::

## Conclusiones

- **P1 (nacionalidad).** La población con nacionalidad española crece 2,92 millones entre 2002 y 2025 (+7 %), pero apenas se mueve desde 2016 (en torno a 42 millones). La extranjera se multiplica por cuatro (de 1,74 a 6,91 millones; del 4,2 % al 14,1 % del total), con un ciclo claro: crece hasta 2010, cae hasta 2016–2017 y repunta, sobre todo desde 2023. Explica el 63,9 % del aumento del total entre 2002 y 2025, el 72,3 % en 2002–2008 y el 93,3 % en 2019–2025.
- **P2 (país de origen).** Los nacidos en el extranjero pasan de 2,33 a 9,46 millones (del 5,7 % al 19,3 %) y explican el 88,1 % del aumento del total. Los nacidos en España alcanzan su máximo en 2013 (40,55 millones) y pierden 882.928 personas hasta 2025. Por regiones, Sudamérica es la principal región de nacimiento (39,5 % en 2025) y la que más aporta al crecimiento reciente (54,6 % en 2019–2025).
- **Nacionalidad frente a nacimiento.** «Extranjero» no significa lo mismo en las dos definiciones: en 2025 hay 2,55 millones más de nacidos en el extranjero que de personas con nacionalidad extranjera, y la diferencia crece todos los años. Según la definición, el peso va del 14,1 % al 19,3 % y el papel de la inmigración en el crecimiento, del 63,9 % al 88,1 %.

## Explicaciones y marco teórico

*(Pendiente: cada apartado con referencias `[@...]`. Distinguir lo que muestran los datos de lo que es interpretación.)*

- **Dinámica demográfica.** Saldo natural, envejecimiento y papel de la migración en el crecimiento.
- **Migración laboral y ciclo económico.** Atracción por demanda de trabajo; efecto de la crisis de 2008 en llegadas, salidas y retornos.
- **Redes migratorias.** Composición por origen y persistencia de los flujos.
- **Marco jurídico.** Adquisición de la nacionalidad española y su efecto sobre la clasificación española/extranjera; regularizaciones y su efecto en el padrón.
- **Medición.** Cómo mide el INE la población (ECP, base censal, revisiones).

## Limitaciones

- Son **stocks, no flujos**: la variación mezcla llegadas, salidas, nacimientos, defunciones y cambios de nacionalidad.
- Los **cambios de nacionalidad** pasan población de «extranjera» a «española» sin que nadie se mueva.
- «Nacido en el extranjero» incluye a españoles nacidos fuera; no equivale a «llegada reciente».
- Posible **ruptura de serie** alrededor de 2021 (transición a la ECP y revisión censal) y cambio de clasificaciones de Europa en 2020. En el total no hay un salto aislado en 2020–2021 (+399.099, +82.748 y +85.929 en 2020, 2021 y 2022), pero el aumento de 2023 (+598.634) es el mayor desde 2008 y, con estos datos, no se puede descartar que parte de él refleje cambios de método o de registro; conviene contrastarlo con otras fuentes.
- Datos hasta el 1-I-2025; solo nivel nacional.
- Análisis **descriptivo**: no atribuye causas.

## Referencias

<!-- Correcciones: AAAA-MM-DD · qué se corrigió -->